# PixelCNN from Scratch: Autoregressive Image Generation on CIFAR-10

> **Notebook goal:** provide a practical reference for learning PixelCNN in PyTorch—from probability factorization and causal masks through residual blocks and training to result analysis, sampling, and discretized logistic mixtures.

PixelCNN models an image probability as a product of conditional pixel probabilities. It is an **explicit-density** model: it assigns observed images a likelihood. The trade-off for this straightforward objective is slow, sequential generation.

This notebook uses PyTorch, torchvision, NumPy, and Matplotlib. torchvision downloads CIFAR-10 on the first run. The code selects a CUDA GPU when available and otherwise runs on CPU. Training the full model may take a while; configuration options let you start with a shorter run.

**Contents**
1. Setup, data, and visualization
2. Autoregressive factorization and causality
3. Masked convolutions and a leakage check
4. Residual blocks and the PixelCNN architecture
5. Discretized logistic mixture (DML) for RGB
6. Training, generation, and analysis
7. Limitations, experiments, and next steps


## 1. Environment setup

If any libraries are missing, uncomment the install command in a separate cell. In a typical PyTorch environment, install the `torch` and `torchvision` versions appropriate for your platform from the [official PyTorch page](https://pytorch.org/get-started/locally/).


In [ ]:
import math
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


## 2. CIFAR-10: images and representation

CIFAR-10 contains 32×32 color images from ten classes. Labels are not fed into the model: we learn an **unconditional** image distribution. Images remain 8-bit values, represented as floating-point numbers in `[0, 1]`. We do not standardize channels by their mean and standard deviation, because the model will represent each channel's discrete intensity directly.

The validation set is held out from the official training set. We reserve the CIFAR-10 test set for final evaluation, so we can compare variants without tuning on the test data.


In [ ]:
DATA_ROOT = Path("data")
BATCH_SIZE = 64
NUM_WORKERS = 0  # Safe default for notebooks on Windows
LIMIT_TRAIN = None  # e.g. 5000 for a quick workflow check
LIMIT_VAL = 2000

transform = transforms.ToTensor()  # uint8 -> float w [0, 1]
train_full = datasets.CIFAR10(DATA_ROOT, train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=transform)

# Fixed validation split; the generator ensures reproducibility.
gen = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(train_full), generator=gen).tolist()
val_indices = perm[:LIMIT_VAL]
train_indices = perm[LIMIT_VAL:]
if LIMIT_TRAIN is not None:
    train_indices = train_indices[:LIMIT_TRAIN]
train_dataset = Subset(train_full, train_indices)
val_dataset = Subset(train_full, val_indices)

loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

class_names = train_full.classes
print(f"train={len(train_dataset):,}, val={len(val_dataset):,}, test={len(test_dataset):,}")


In [ ]:
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, label in zip(axes.flat, images[:10], labels[:10]):
    ax.imshow(image.permute(1, 2, 0))
    ax.set_title(class_names[label])
    ax.axis("off")
plt.tight_layout()


## 3. Autoregressive factorization and pixel order

For an image with `H×W` pixels, we use raster order: left to right, top to bottom. Then

`p(x) = ∏ᵢ p(xᵢ | x₁, …, xᵢ₋₁)`.

A convolutional network can compute conditional distribution parameters at all positions in parallel during training. Masks ensure that the prediction at position `i` cannot use future pixels. During generation, future values are unknown, so we must fill the image one position at a time.

For RGB, we model the **color vector of each pixel** conditioned on earlier pixels. Within that vector, a logistic mixture introduces channel dependencies: `G` depends on `R`, and `B` depends on both `R` and `G`.


## 4. Masked convolutional layers

A **type A** mask zeros the filter center and the entire future region. We use it in the first layer, whose input consists of observed pixels. A **type B** mask allows the center while still blocking future positions; we use it on hidden representations. This lets information at the current position flow through the network without leaking the input pixel into its own prediction.

This mask implementation also handles multi-layer convolutions (dilation and kernel size), rather than only the common 3×3 kernel.


In [ ]:
class MaskedConv2d(nn.Conv2d):
    """Causal convolution for raster order (type A or B)."""
    def __init__(self, *args, mask_type="B", **kwargs):
        if mask_type not in {"A", "B"}:
            raise ValueError("mask_type must be either 'A' or 'B'")
        super().__init__(*args, **kwargs)
        kh, kw = self.kernel_size
        if kh % 2 == 0 or kw % 2 == 0:
            raise ValueError("This implementation requires odd-sized kernels")
        mask = torch.ones_like(self.weight)
        cy, cx = kh // 2, kw // 2
        mask[:, :, cy + 1 :, :] = 0
        mask[:, :, cy, cx + 1 :] = 0
        if mask_type == "A":
            mask[:, :, cy, cx] = 0
        self.register_buffer("mask", mask)

    def forward(self, x):
        return F.conv2d(
            x, self.weight * self.mask, self.bias,
            self.stride, self.padding, self.dilation, self.groups
        )


In [ ]:
# Causality check: changing pixel (y, x) must not change outputs
# at this position or at any earlier position in raster order.
layer = MaskedConv2d(3, 4, kernel_size=3, padding=1, mask_type="A")
x1 = torch.randn(1, 3, 5, 5)
x2 = x1.clone()
x2[:, :, 2, 2] += 100
with torch.no_grad():
    y1, y2 = layer(x1), layer(x2)
# Positions before (2, 2): all rows < 2 and columns < 2 in row 2.
assert torch.allclose(y1[:, :, :2, :], y2[:, :, :2, :])
assert torch.allclose(y1[:, :, 2, :2], y2[:, :, 2, :2])
assert torch.allclose(y1[:, :, 2, 2], y2[:, :, 2, 2])  # type A mask blocks the center
print("Type A mask blocks the current and future pixels.")


## 5. Residual block and receptive field

Residual blocks make deeper networks easier to optimize: they learn corrections to the input representation instead of reconstructing the whole representation from scratch. Every convolution in the block's path uses a type B mask. Pointwise normalization and activation do not mix spatial positions, so they preserve causality.

Here we use `1×1 → 3×3 → 1×1`. The 3×3 filter expands the receptive field by one pixel in each direction per block. Practical models also use more layers, dilation, or separate vertical and horizontal stacks to cover more context efficiently.


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels, hidden_channels=None):
        super().__init__()
        hidden_channels = hidden_channels or channels
        self.net = nn.Sequential(
            nn.ReLU(inplace=True),
            MaskedConv2d(channels, hidden_channels, kernel_size=1, mask_type="B"),
            nn.ReLU(inplace=True),
            MaskedConv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1, mask_type="B"),
            nn.ReLU(inplace=True),
            MaskedConv2d(hidden_channels, channels, kernel_size=1, mask_type="B"),
        )

    def forward(self, x):
        return x + self.net(x)


## 6. Discretized logistic mixture for RGB

Image channels are discrete: `0, …, 255`. Instead of predicting 256 classes independently for every channel, we use a **mixture of discretized logistic distributions**. For each pixel, the model emits `K` mixture weights, RGB means and scales, and color-dependency coefficients. The logistic component is continuous, but its density is integrated over the interval corresponding to each 8-bit value.

The parameters for channels G and B are adjusted using channels sampled earlier:

- `μG ← μG + aRG·R`
- `μB ← μB + aRB·R + aGB·G`

The RGB log-likelihoods within a pixel are summed conditionally, then mixture components are combined with `logsumexp`. We minimize negative log-likelihood in **nats** and report bits per dimension (bits/dim) as `NLL / (H·W·C·ln 2)`.


In [ ]:
def discretized_mix_logistic_loss(params, x, num_mixtures):
    """Negative log-likelihood; params: (B, K*10, H, W), x in [-1, 1]."""
    b, _, h, w = params.shape
    params = params.permute(0, 2, 3, 1).contiguous()
    logit_probs = params[..., :num_mixtures]
    values = params[..., num_mixtures:].view(b, h, w, 3, 3 * num_mixtures)
    means = values[..., :num_mixtures]
    log_scales = torch.clamp(values[..., num_mixtures:2*num_mixtures], min=-7.0)
    coeffs = torch.tanh(values[..., 2*num_mixtures:])

    x = x.permute(0, 2, 3, 1).contiguous()  # input is already in [-1, 1]
    xr, xg, xb = x.unbind(dim=-1)
    mr, mg, mb = means.unbind(dim=-2)
    sr, sg, sb = log_scales.unbind(dim=-2)
    crg, crb, cgb = coeffs.unbind(dim=-2)

    xr = xr.unsqueeze(-1)
    xg = xg.unsqueeze(-1)
    xb = xb.unsqueeze(-1)
    m_r = mr
    m_g = mg + crg * xr
    m_b = mb + crb * xr + cgb * xg

    inv_scale_r, inv_scale_g, inv_scale_b = torch.exp(-sr), torch.exp(-sg), torch.exp(-sb)
    half_bin = 1.0 / 255.0

    def channel_log_prob(xc, mean, log_scale, inv_scale):
        centered = xc - mean
        plus = inv_scale * (centered + half_bin)
        minus = inv_scale * (centered - half_bin)
        cdf_plus = torch.sigmoid(plus)
        cdf_minus = torch.sigmoid(minus)
        log_cdf_plus = F.logsigmoid(plus)
        log_one_minus_cdf_minus = F.logsigmoid(-minus)
        cdf_delta = cdf_plus - cdf_minus
        mid = inv_scale * centered
        log_pdf_mid = mid - log_scale - 2.0 * F.softplus(mid)
        log_cdf_delta = torch.where(
            cdf_delta > 1e-5,
            torch.log(torch.clamp(cdf_delta, min=1e-12)),
            log_pdf_mid + math.log(2.0 * half_bin),
        )
        log_prob = torch.where(xc < -0.999, log_cdf_plus,
                               torch.where(xc > 0.999, log_one_minus_cdf_minus, log_cdf_delta))
        return log_prob

    log_probs = (
        channel_log_prob(xr, m_r, sr, inv_scale_r)
        + channel_log_prob(xg, m_g, sg, inv_scale_g)
        + channel_log_prob(xb, m_b, sb, inv_scale_b)
    )
    log_prob = torch.logsumexp(F.log_softmax(logit_probs, dim=-1) + log_probs, dim=-1)
    return -log_prob.mean()


## 7. PixelCNN: the prediction network

The first type A layer ensures that the input signal excludes the current target pixel. Subsequent type B blocks can combine information at the current position with its hidden state, which is already causal. A `1×1` head emits mixture parameters at each position. With `K` components, the output has `K + 3×3K = 10K` channels.

The entire network remains causal only if every spatial operation is causal. Do not add an ordinary 3×3 convolution or normalization across spatial positions without checking whether it can expose future pixels.


In [ ]:
class PixelCNN(nn.Module):
    def __init__(self, channels=64, num_residual=6, num_mixtures=10):
        super().__init__()
        self.num_mixtures = num_mixtures
        self.input_conv = MaskedConv2d(3, channels, kernel_size=7, padding=3, mask_type="A")
        self.residual_blocks = nn.Sequential(*[ResidualBlock(channels) for _ in range(num_residual)])
        self.output_net = nn.Sequential(
            nn.ReLU(inplace=True),
            MaskedConv2d(channels, channels, kernel_size=1, mask_type="B"),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, 10 * num_mixtures, kernel_size=1),
        )

    def forward(self, x):
        x = x * 2.0 - 1.0  # model values in [-1, 1]
        return self.output_net(self.residual_blocks(self.input_conv(x)))

model = PixelCNN(channels=64, num_residual=6, num_mixtures=10).to(device)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {num_params:,}")
with torch.no_grad():
    dummy = torch.zeros(2, 3, 32, 32, device=device)
    out = model(dummy)
print("output:", tuple(out.shape), "| expected channels:", 10 * model.num_mixtures)


## 8. Training and metrics

Training is fully parallel across positions: for each image, we compute the log-probability of all pixels in one pass. Bits/dim makes likelihood comparable regardless of the number of channels or image size. Lower is better.

The configuration below is a starting point, not a guarantee of the best result. Quality depends on training time, architecture, and hardware. First run a few epochs to check the workflow, then increase the budget.


In [ ]:
EPOCHS = 10
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-5
MAX_GRAD_NORM = 1.0
CHECKPOINT_PATH = Path("pixelcnn_cifar10.pt")

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

def bits_per_dim(nll_nats):
    # The loss averages over spatial positions (H, W) and sums over RGB channels.
    return nll_nats / (3 * math.log(2.0))

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_nll, total_examples = 0.0, 0
    for x, _ in loader:
        x = x.to(device, non_blocking=True)
        params = model(x)
        nll = discretized_mix_logistic_loss(params, x * 2 - 1, model.num_mixtures)
        total_nll += nll.item() * x.size(0)
        total_examples += x.size(0)
    return total_nll / total_examples

def train_one_epoch(model, loader):
    model.train()
    total_nll, total_examples = 0.0, 0
    for step, (x, _) in enumerate(loader, start=1):
        x = x.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        params = model(x)
        loss = discretized_mix_logistic_loss(params, x * 2 - 1, model.num_mixtures)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        total_nll += loss.item() * x.size(0)
        total_examples += x.size(0)
        if step % 100 == 0:
            print(f"  batch {step:4d}/{len(loader)} | nll={loss.item():.3f} nats/RGB pixel")
    return total_nll / total_examples


In [ ]:
history = {"train_bpd": [], "val_bpd": []}
best_val = float("inf")

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_nll = train_one_epoch(model, train_loader)
    val_nll = evaluate(model, val_loader)
    history["train_bpd"].append(bits_per_dim(train_nll))
    history["val_bpd"].append(bits_per_dim(val_nll))
    elapsed = time.time() - start
    print(f"Epoch {epoch:02d}/{EPOCHS} | train {history['train_bpd'][-1]:.4f} bpd | "
          f"val {history['val_bpd'][-1]:.4f} bpd | {elapsed/60:.1f} min")
    if val_nll < best_val:
        best_val = val_nll
        torch.save({"model": model.state_dict(), "config": {
            "channels": 64, "num_residual": 6, "num_mixtures": model.num_mixtures,
            "epoch": epoch, "val_bpd": bits_per_dim(val_nll),
        }}, CHECKPOINT_PATH)
        print(f"  saved best checkpoint: {CHECKPOINT_PATH}")


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history["train_bpd"], marker="o", label="train")
plt.plot(history["val_bpd"], marker="o", label="validation")
plt.xlabel("Epoch")
plt.ylabel("Bits per dimension (lower is better)")
plt.title("PixelCNN on CIFAR-10")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


### How to read the curves

- Falling training and validation bpd indicate that the model is learning image structure.
- Falling training bpd with rising validation bpd suggests overfitting; consider a smaller model, augmentation (while preserving the correct density interpretation), or regularization.
- High bpd after a short run does not necessarily indicate a bug: CIFAR-10 is diverse, and PixelCNN needs many updates.
- To compare published results, check the exact data variant, likelihood, preprocessing, and reporting method. A number without its evaluation protocol can be misleading.


## 9. Sampling: from distribution to image

Sampling is sequential. For each pixel, we compute a prediction, draw a mixture component, and then draw RGB channels from logistic distributions. This simple code runs the entire network again for every pixel, so it is very slow. It is a clear reference implementation; efficient PixelCNN implementations cache activations or use optimized kernels.

We sample a mixture component using its softmax weights. Logistic sampling uses the inverse CDF applied to a uniform random variable. Values are clipped to the 8-bit range and quantized at the end.


In [ ]:
@torch.no_grad()
def sample_logistic(mean, log_scale):
    u = torch.rand_like(mean).clamp_(1e-5, 1 - 1e-5)
    return mean + torch.exp(log_scale) * (torch.log(u) - torch.log1p(-u))

@torch.no_grad()
def sample_pixelcnn(model, n_samples=4, height=32, width=32, temperature=1.0):
    model.eval()
    x = torch.zeros(n_samples, 3, height, width, device=device)
    k = model.num_mixtures
    for row in range(height):
        for col in range(width):
            params = model(x)
            p = params[:, :, row, col]
            logits = p[:, :k] / temperature
            values = p[:, k:].view(n_samples, 3, 3 * k)
            means, log_scales, coeffs = values[:, :, :k], values[:, :, k:2*k], torch.tanh(values[:, :, 2*k:])
            log_scales = log_scales.clamp(min=-7.0)
            mixture = torch.distributions.Categorical(logits=logits).sample()
            gather_index = mixture[:, None, None].expand(-1, 3, 1)
            means = means.gather(-1, gather_index).squeeze(-1)
            log_scales = log_scales.gather(-1, gather_index).squeeze(-1)
            coeffs = coeffs.gather(-1, gather_index).squeeze(-1)

            r = sample_logistic(means[:, 0], log_scales[:, 0]).clamp(-1, 1)
            g_mean = means[:, 1] + coeffs[:, 0] * r
            g = sample_logistic(g_mean, log_scales[:, 1]).clamp(-1, 1)
            b_mean = means[:, 2] + coeffs[:, 1] * r + coeffs[:, 2] * g
            blue = sample_logistic(b_mean, log_scales[:, 2]).clamp(-1, 1)
            pixel = torch.stack([r, g, blue], dim=1)
            pixel = torch.round((pixel + 1) * 127.5).clamp(0, 255) / 255.0
            x[:, :, row, col] = pixel
    return x


In [ ]:
# Load the best checkpoint if training has already been run.
if CHECKPOINT_PATH.exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint["model"])
    print("Loaded epoch", checkpoint["config"]["epoch"],
          "| val bpd", round(checkpoint["config"]["val_bpd"], 4))
else:
    print("No checkpoint found — samples will use the current model weights.")

samples = sample_pixelcnn(model, n_samples=4)
grid = make_grid(samples.cpu(), nrow=2, padding=2)
plt.figure(figsize=(6, 6))
plt.imshow(grid.permute(1, 2, 0))
plt.axis("off")
plt.title("PixelCNN samples")
plt.show()


## 10. Model analysis and failure modes

### 10.1 Test-set log-likelihood

Use the test set only after making design decisions. We report bpd averaged over images.


In [ ]:
test_nll = evaluate(model, test_loader)
print(f"CIFAR-10 test: {bits_per_dim(test_nll):.4f} bits/dim")


### 10.2 Comparing real and generated images

PixelCNN is not an autoencoder and does not directly reconstruct images. The visualization below shows real images next to generated samples. Look at local textures, shape coherence, and recurring artifacts. A low NLL does not guarantee the most visually appealing samples: likelihood and perceptual quality measure different things.


In [ ]:
real, _ = next(iter(test_loader))
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i in range(4):
    axes[0, i].imshow(real[i].permute(1, 2, 0)); axes[0, i].set_title("CIFAR-10"); axes[0, i].axis("off")
    axes[1, i].imshow(samples[i].cpu().permute(1, 2, 0)); axes[1, i].set_title("PixelCNN"); axes[1, i].axis("off")
plt.tight_layout()


### 10.3 Generation diagnostics

Inspect whether (1) samples have realistic local textures, (2) objects retain coherent shapes, (3) checkerboard patterns or stripes appear, and (4) samples are diverse. Weak global structure alongside plausible local patches often points to a receptive field that is too small or insufficient training.

You can also compare sampling temperatures. A value below 1 usually produces more conservative but less diverse samples. Temperature affects generation, not the computed NLL.


## 11. Key takeaways

1. **Autoregression** gives an exact probability factorization, but imposes a generation order.
2. **Type A masks** hide the current pixel from its prediction; **type B masks** let hidden features use the current position because those features are already causal.
3. **Residual blocks** support deeper networks and improve gradient flow.
4. **Mixtures of logistics** model discrete intensity and channel dependencies more efficiently than an independent 256-class softmax for each RGB channel.
5. **Parallel training, sequential generation** is the key practical distinction from many other generative models.
6. **BPD** is a useful quantitative metric, but inspect samples and check the evaluation protocol too.

### Limitations of this implementation

- Slow sampling because the full network runs once per pixel.
- A single masked 2D convolution does not use separate vertical and horizontal streams; this is a simpler PixelCNN variant.
- Short training and limited width/depth will constrain sample quality.
- No augmentation or class conditioning.
- Computing a difference of CDFs can be numerically difficult; this implementation uses a density approximation for narrow intervals and stable log-sigmoid operations.

## 12. Further exercises

1. **Mask and order:** change to column-major order. Update the mask and causality check.
2. **Broader context:** add dilated masked convolutions. How do receptive field and bpd change?
3. **Class conditioning:** add a CIFAR-10 label embedding to residual blocks (for example, as a channel-wise bias). Compare samples for selected classes.
4. **Model size:** compare 3, 6, and 12 residual blocks under a similar training budget. Track learning curves and cost.
5. **Number of mixtures:** compare `K=1, 5, 10`. How do cost and likelihood change?
6. **Faster inference:** cache activations in raster order or implement a specialized kernel. Measure speedup over `sample_pixelcnn`.
7. **Architecture comparison:** compare the baseline PixelCNN with gated PixelCNN or PixelCNN++ and describe differences in receptive field and likelihood.
8. **Failure analysis:** show samples at several temperatures and the test examples with the lowest log-likelihood.

### Further reading

- van den Oord et al., [Conditional Image Generation with PixelCNN Decoders (2016)](https://arxiv.org/abs/1606.05328)
- Salimans et al., [PixelCNN++: Improving the PixelCNN with Discretized Logistic Mixture Likelihood and Other Modifications (2017)](https://arxiv.org/abs/1701.05517)
- van den Oord et al., [Pixel Recurrent Neural Networks (2016)](https://arxiv.org/abs/1601.06759)
